In [ ]:
%%sql -r dataframe_1
USE DATABASE DISNEYLAND;
USE SCHEMA REVIEWS;

SELECT *
FROM GUEST_REVIEWS
LIMIT 100

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE GUEST_REVIEWS_ENRICHED
AS

SELECT *, SNOWFLAKE.CORTEX.TRANSLATE(REVIEW_TEXT, '', 'en') AS review_english
FROM GUEST_REVIEWS

In [ ]:
%%sql -r dataframe_3
SELECT *
FROM GUEST_REVIEWS_ENRICHED

In [ ]:
%%sql -r dataframe_4
ALTER TABLE GUEST_REVIEWS_ENRICHED
ADD COLUMN review_summary VARCHAR;

In [ ]:
%%sql -r dataframe_5
UPDATE GUEST_REVIEWS_ENRICHED
SET review_summary = SNOWFLAKE.CORTEX.SUMMARIZE(review_english)

In [ ]:
%%sql -r dataframe_6
SELECT REVIEW_ENGLISH, review_summary
FROM GUEST_REVIEWS_ENRICHED
LIMIT 100

In [ ]:
%%sql -r dataframe_7
ALTER TABLE GUEST_REVIEWS_ENRICHED
ADD COLUMN sentiment_value FLOAT;

UPDATE GUEST_REVIEWS_ENRICHED
SET sentiment_value = SNOWFLAKE.CORTEX.SENTIMENT(review_english)

In [ ]:
%%sql -r dataframe_8
SELECT * REVIEW_SUMMARY, SENTIMENT_VALUE
FROM GUEST_REVIEWS_ENRICHED

In [ ]:
%%sql -r dataframe_9
SELECT
    AI_AGG(
    REVIEW_ENGLISH,
    'What are the 6 most common categories in these reviews?'
  ) AS TOP_CATEGORIES
FROM GUEST_REVIEWS_ENRICHED
LIMIT 1000

In [ ]:
%%sql -r dataframe_10
SELECT
    AI_AGG(
    REVIEW_ENGLISH,
    'What are the 6 most common complaints in these reviews?'
  ) AS TOP_CATEGORIES
FROM GUEST_REVIEWS_ENRICHED
WHERE SENTIMENT_VALUE < -0.3
LIMIT 100

In [ ]:
%%sql -r dataframe_11
SELECT
    AI_AGG(
    REVIEW_ENGLISH,
    'Return the 6 most common complaints categories.
    
    Format strictly as :
    0. <category>
    1. <category>
    2. <category>
    3. <category>
    4. <category>
    5. <category>
    
    Use short category names only (2-4 words).
    No explainations, no extra texts.'
  ) AS TOP_CATEGORIES
FROM GUEST_REVIEWS_ENRICHED
WHERE SENTIMENT_VALUE < -0.3
LIMIT 100;

In [ ]:
%%sql -r dataframe_12
SELECT
    AI_AGG(
    REVIEW_ENGLISH,
    'What are the 6 most common positive experiences in these reviews?'
  ) AS TOP_CATEGORIES
FROM GUEST_REVIEWS_ENRICHED
WHERE SENTIMENT_VALUE > 0.3
LIMIT 100

In [ ]:
%%sql -r dataframe_13
SELECT
    AI_AGG(
    REVIEW_ENGLISH,
    'Return the 6 most common categories.
    
    Format strictly as :
    0. <category>
    1. <category>
    2. <category>
    3. <category>
    4. <category>
    5. <category>
    
    Use short category names only (2-4 words).
    No explainations, no extra texts.'
  ) AS TOP_CATEGORIES
FROM GUEST_REVIEWS_ENRICHED
WHERE SENTIMENT_VALUE > 0.3
LIMIT 100;

In [ ]:
%%sql -r dataframe_14
SELECT
REVIEW_SUMMARY,
SNOWFLAKE.CORTEX.CLASSIFY_TEXT(REVIEW_ENGLISH, ['Overcrowding & Queues', 'Staff & Service', 'Pricing & Value', 'Food &
Dining', 'Rides & Attractions', 'Atmosphere & Magic', 'Cleanliness & Maintenance', 'Events & Entertainment']).
FROM GUEST_REVIEWS_ENRICHED
LIMIT 100

In [ ]:
%%sql -r dataframe_15
SELECT
REVIEW_SUMMARY,
SNOWFLAKE.CORTEX.AI_CLASSIFY(REVIEW_ENGLISH, ['Overcrowding & Queues', 'Staff & Service', 'Pricing & Value', 'Food &
Dining', 'Rides & Attractions', 'Atmosphere & Magic', 'Cleanliness & Maintenance', 'Events & Entertainment'],
{'outputmode': 'multi'}
)
FROM GUEST_REVIEWS_ENRICHED
LIMIT 100

In [ ]:
%%sql -r dataframe_16
SELECT
REVIEW_SUMMARY,
SNOWFLAKE.CORTEX.AI_SENTIMENT(REVIEW_ENGLISH, ['Overcrowding & Queues', 'Staff & Service', 'Pricing & Value', 'Food &
Dining', 'Rides & Attractions', 'Atmosphere & Magic', 'Cleanliness & Maintenance', 'Events & Entertainment']
)
FROM GUEST_REVIEWS_ENRICHED
LIMIT 100

In [ ]:
%%sql -r dataframe_17
SELECT
    r.REVIEW_SUMMARY,
    f.value.['name']::VARCHAR AS category,
    f.value.['sentiment']::VARCHAR AS sentiment
FROM GUEST_REVIEWS_ENRICHED r,
LATERAL FLATTEN(
    input => SNOWFLAKE.CORTEX.AI_SENTIMENT(
        r.REVIEW_ENGLISH,
        ['Overcrowding & Queues', 'Staff & Service', 'Pricing & Value',
         'Food & Dining', 'Rides & Attractions', 'Atmosphere & Magic',
         'Cleanliness & Maintenance', 'Events & Entertainment']
    )['categories']
) f
LIMIT 100:

In [ ]:
%%sql -r dataframe_18
CREATE OR REPLACE TABLE GUEST_REVIEWS_ENRICHED_CATEGORIES AS

SELECT
    r.REVIEW_ID,
    r.REVIEW_SUMMARY,
    f.value.['name']::VARCHAR AS category,
    f.value.['sentiment']::VARCHAR AS sentiment
FROM GUEST_REVIEWS_ENRICHED r,
LATERAL FLATTEN(
    input => SNOWFLAKE.CORTEX.AI_SENTIMENT(
        r.REVIEW_ENGLISH,
        ['Overcrowding & Queues', 'Staff & Service', 'Pricing & Value',
         'Food & Dining', 'Rides & Attractions', 'Atmosphere & Magic',
         'Cleanliness & Maintenance', 'Events & Entertainment']
    )['categories']
) f
;

In [ ]:
%%sql -r dataframe_19
SELECT CATEGORY, SENTIMENT, COUNT(REVIEW_ID) AS REVIEW_COUNT
FROM GUEST_REVIEWS_ENRICHED_CATEGORIES
GROUP BY CATEGORY, SENTIMENT